In [1]:
# reactive/agent.py
import json

def handle_triage_reactive(patient_data: dict) -> dict:
    """
    Reactive Architecture: Rules-based pure logic loop.
    No model calls, strict hardcoded conditions.
    """
    fault_score = patient_data.get("triage_score", 0)
    has_allergy = patient_data.get("critical_allergy", False)
    bed_available = patient_data.get("icu_bed_available", False)
    
    # Rule 1: High severity requiring ICU
    if fault_score >= 8:
        if bed_available:
            return {"action": "ASSIGN_ICU", "status": "SUCCESS", "reason": "High priority score with available bed."}
        else:
            return {"action": "ESCALATE_HUMAN", "status": "WAITING", "reason": "High priority but NO bed available."}
    
    # Rule 2: Moderate severity needing surgery/observation
    elif 5 <= fault_score < 8:
        if has_allergy:
            return {"action": "FLAG_ANESTHESIA", "status": "PENDING_REVIEW", "reason": "Moderate score with critical allergy flag."}
        return {"action": "ASSIGN_REGULAR_BED", "status": "SUCCESS", "reason": "Standard admission."}
    
    # Rule 3: Low severity
    else:
        return {"action": "DISCHARGE_OUTPATIENT", "status": "COMPLETED", "reason": "Non-critical condition."}

if __name__ == "__main__":
    # Test Case
    sample_patient = {"triage_score": 9, "critical_allergy": True, "icu_bed_available": False}
    result = handle_triage_reactive(sample_patient)
    print("Reactive Agent Output:", json.dumps(result, indent=2))

Reactive Agent Output: {
  "action": "ESCALATE_HUMAN",
  "status": "WAITING",
  "reason": "High priority but NO bed available."
}


In [2]:
pip install google-genai

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ------------------------------ --------- 0.8/1.0 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 2.5 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.8 MB 2.4 MB/s eta 0:00:02
   ----------- ---------------------------- 1.0/3.8 MB 2.4 MB/s eta 0:00:02
   ---------------- ----------------------- 1.6/3.8 MB 2.5 MB/s eta 0:00:01
   ---------------------- ----------------- 2.1/3.8 MB 2.7 MB/s eta 0:00:01
   --------------------------- ------------ 2.6/3.8 MB 2.5 MB/s eta 0:00:01
   ------------------------------ --------- 2.9/3.8 MB 2.3 MB/s eta 0:00:01
   ----------------------------------- ---- 3.4/3.8 MB 2.3 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 2.3 MB/s  0:00:01

   -----------------------------------


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. تحميل متغيرات البيئة من ملف .env في جذر المشروع
load_dotenv()

# 2. إنشاء الـ Client (سيقوم تلقائياً بقراءة GEMINI_API_KEY من البيئة)
client = genai.Client(api_key="AQ.Ab8RN6KBQXgrothLWJTbhpueRd-6vghw5iidH5zD58kbAYsIAw")

# --- Tools Definition ---
def check_icu_beds() -> str:
    """Checks real-time ICU bed capacity."""
    return "ICU Bed #3 is currently available, ICU Bed #4 is under maintenance."

def get_patient_history(patient_id: str) -> str:
    """Fetches patient medical history and allergies."""
    return f"Patient {patient_id}: Severe penicillin allergy, prior cardiac arrest in 2021."

tools = [check_icu_beds, get_patient_history]

# --- Agent Core Logic ---
def run_unconstrained_agent(prompt: str):
    print("--- Unconstrained ReAct Loop Starting ---")
    
    # استدعاء الموديل مع تحديد اسم الموديل الصح والـ Tools
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=tools,
            temperature=0.2,
            system_instruction=(
                "You are an unconstrained emergency triage AI agent. "
                "You can call tools freely to resolve ICU/Surgery room allocations."
            )
        )
    )
    
    # التحقق مما إذا كان الموديل يريد استدعاء أداة (Tool Call)
    if response.function_calls:
        for call in response.function_calls:
            print(f"[LLM Decision]: Executing Tool -> {call.name} with args {call.args}")
            
            # تنفيذ الأدوات ديناميكياً
            if call.name == "check_icu_beds":
                res = check_icu_beds()
            elif call.name == "get_patient_history":
                res = get_patient_history(call.args.get("patient_id", "P-101"))
            else:
                res = "Tool not found."
            
            print(f"[Tool Result]: {res}")
    else:
        print(f"[LLM Response]: {response.text}")

if __name__ == "__main__":
    run_unconstrained_agent("Patient P-101 arrived with severe chest pain and triage score 9. What should we do?")

--- Unconstrained ReAct Loop Starting ---


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 34.339348236s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.go